# STAC Basics Exercise - ANSWER KEY

This is the complete answer key for the STAC basics exercise. This notebook contains all the solutions to the fill-in-the-blank exercises.

## Learning Objectives

By the end of this exercise, you will be able to:

- Understand the core components of STAC: Catalogs, Collections, Items, and Assets
- Connect to a STAC API and browse available data
- Search for geospatial data using spatial and temporal filters
- Explore metadata associated with STAC items
- Visualize geospatial data on interactive maps
- Complete practical exercises to reinforce your learning

**Let's get started!**

## 1. Install Required Libraries

Before we begin, we need to install the necessary Python libraries for working with STAC data. Run the cell below to install all required packages.

In [ ]:
# Install required libraries
!pip install pystac-client requests folium matplotlib pillow

## 2. Understanding STAC Components

STAC is a specification for describing geospatial information. It provides a common language to describe a range of geospatial information, making it easier to index and discover satellite imagery and other geospatial data.

STAC organizes geospatial data in a hierarchical structure with four main components:

### Catalog
- The root of the STAC structure
- Contains links to collections and other catalogs
- Provides high-level organization of data

### Collection
- A group of related items (e.g., all Sentinel-2 images)
- Contains metadata about the dataset as a whole
- Defines common properties shared by all items

### Item
- A single geospatial asset (e.g., one satellite image)
- Contains metadata like datetime, geometry, and properties
- Points to one or more assets

### Asset
- The actual data files (e.g., GeoTIFF image, thumbnail)
- Referenced by items but stored separately
- Can include different formats and processing levels

## 3. Connecting to a STAC API

Now let's connect to a real STAC API! We'll use **EarthSearch**, which provides free access to a vast collection of geospatial datasets including satellite imagery from AWS Open Data.

The EarthSearch STAC API contains datasets like:
- Sentinel-2 (optical imagery)
- Landsat (long-term optical imagery)
- Sentinel-1 (SAR imagery)
- And many more open datasets!

In [ ]:
# Import STAC Python Client library
import pystac_client

In [ ]:
# Connect to the EarthSearch STAC API
api_url = "https://earth-search.aws.element84.com/v1"
catalog = pystac_client.Client.open(api_url)

print("Connected to EarthSearch STAC API!")
print(f"Catalog Title: {catalog.title}")
print(f"Description: {catalog.description}")

# List some available collections
collections = list(catalog.get_collections())
print(f"Total Collections Available: {len(collections)}")
print("Sample Collections:")
for i, collection in enumerate(collections[:5]):
    print(f"  {i+1}. {collection.id} - {collection.title}")

## 4. Searching for Items

One of the most powerful features of STAC is the ability to search for data based on:

- **Spatial extent** (bounding box or geometry)
- **Temporal range** (date/time filters)
- **Collection type** (e.g., Sentinel-2, Landsat)
- **Properties** (cloud cover, platform, etc.)

Let's search for Sentinel-2 images over Enschede, Netherlands:

In [ ]:
# Execute the search
search = catalog.search(
    collections=["sentinel-2-l2a"],      # Sentinel-2 L2A collection
    bbox=[6.8, 52.2, 6.9, 52.3],         # Enschede, Netherlands
    datetime="2025-06-01/2025-06-30",    # June 2025
    max_items=10,                        # Maximum 10 items
    query={"eo:cloud_cover": {"lt": 20}} # Less than 20% cloud cover
)

items = list(search.items())
print(f"Found {len(items)} items over Enschede, Netherlands.")

# Print details of the first item if available
if items:
    first_item = items[0]
    print(f"First item:")
    print(f"  ID: {first_item.id}")
    print(f"  Date: {first_item.datetime}")
    print(f"  Cloud Cover: {first_item.properties.get('eo:cloud_cover', 'N/A')}%")

else:
    print("No items found. Check your search parameters!")

## 5. Exploring Item Metadata

Each STAC item contains rich metadata that tells us about the geospatial asset. Let's examine what information is available:

In [ ]:
# Explore metadata of the first item
if items:
    item = items[0]
    
    print(f"ID: {item.id}")
    print(f"Date: {item.datetime}")
    print(f"Geometry Type: {item.geometry['type']}")
    print(f"Bounding Box: {item.bbox}")
    print(f"Cloud Cover: {item.properties.get('eo:cloud_cover', 'N/A')}%")
    print(f"Platform: {item.properties.get('platform', 'N/A')}")
    print(f"Instruments: {item.properties.get('instruments', 'N/A')}")

    print(f"Available assets ({len(item.assets)} total):")
    for asset_key, asset in list(item.assets.items())[:8]:  # Show first 8 assets
        title = asset.title if asset.title else asset_key
        print(f"  {asset_key}: {title}")
        print(f"    Type: {asset.media_type}")
        print(f"    Size: {asset.extra_fields.get('file:size', 'Unknown')}")

else:
    print("No items found to explore.")

## 6. Visualizing Assets

Let's create an interactive map to visualize the spatial extent of our STAC items. This helps us understand where the data covers on Earth.

In [ ]:
# Import folium interactive mapping library
import folium

In [ ]:
# Create an interactive map showing item footprints
if items:
    # Calculate overall bounding box from all items
    lats, lons = [], []
    
    for item in items:
        # Extract coordinates from bounding box
        lats.extend([item.bbox[1], item.bbox[3]])
        lons.extend([item.bbox[0], item.bbox[2]])
    
    min_lat, max_lat = min(lats), max(lats)
    min_lon, max_lon = min(lons), max(lons)
    
    # Create the map
    m = folium.Map(tiles='OpenStreetMap')  # OpenStreetMap
    m.fit_bounds([[min_lat, min_lon], [max_lat, max_lon]])  # Fit map to the overall bounds
    
    # Add each item to the map
    for i, item in enumerate(items[:5]):  # Show first 5 items to avoid clutter
        # Create popup text with item information
        popup_text = f"""
        <b>Item {i+1}</b><br>
        ID: {item.id}<br>
        Date: {item.datetime.strftime('%Y-%m-%d')}<br>
        Cloud Cover: {item.properties.get('eo:cloud_cover', 'N/A')}%
        """
        
        # Add geometry to map
        folium.GeoJson(
            item.geometry,
            popup=folium.Popup(popup_text, max_width=300),
            style_function=lambda x, color=f"#{hash(item.id) % 0xFFFFFF:06x}": {
                'fillColor': color,
                'color': color,
                'weight': 2,
                'fillOpacity': 0.3
            }
        ).add_to(m)
    
    # Display map
    display(m)

else:
    print("No items to visualize.")

## 7. Download and Display a Thumbnail

Find and display a thumbnail image from one of the search results.

In [ ]:
# Import libraries
import requests
from PIL import Image
import matplotlib.pyplot as plt

In [ ]:
# Download and display a thumbnail
if items:
    item = items[0]  # Use the first item
    
    # Check if thumbnail asset exists
    if 'thumbnail' in item.assets:
        thumbnail_asset = item.assets['thumbnail']
        thumbnail_url = thumbnail_asset.href
        
        # Download the thumbnail
        print(f"Downloading thumbnail from: {thumbnail_url}")
        response = requests.get(thumbnail_url, stream=True)
        
        if response.status_code == 200:
            # Open image
            img = Image.open(response.raw)
            
            # Display image
            plt.figure(figsize=(8, 8))
            plt.imshow(img)
            plt.title(f"Thumbnail Item: {item.id}")
            plt.axis('off')  # Hide axes
            plt.show()       # Show plot
            
        else:
            print(f"Failed to download thumbnail. Status code: {response.status_code}")
    else:
        print("No thumbnail available for this item.")
else:
    print("No items available.")

# Conclusion

Congratulations, you have successfully completed the STAC basics exercise!

## What You've Learned

- **STAC Structure**: Understanding Catalogs, Collections, Items, and Assets
- **API Connection**: Connecting to real-world STAC APIs
- **Data Discovery**: Searching for geospatial data using spatial and temporal filters
- **Metadata Exploration**: Extracting and analyzing item properties
- **Visualization**: Creating interactive maps with geospatial data
- **Asset Access**: Downloading and displaying satellite imagery

## Next Steps

Now that you understand STAC basics, you can:

1. **Explore other STAC APIs**: Try Earth Search, Radiant Earth, or other providers
2. **Work with different datasets**: Landsat, MODIS, aerial imagery, etc.
3. **Advanced filtering**: Use more complex queries and property filters
4. **Data analysis**: Process downloaded assets for scientific analysis

## Additional Resources

- [STAC Specification](https://stacspec.org/)
- [PySTAC Documentation](https://pystac.readthedocs.io/)
- [EarthSearch](https://radiantearth.github.io/stac-browser/#/external/earth-search.aws.element84.com/v1)

**Happy exploring with STAC!**